In [60]:
def assign_mutype_snpindex(sitemutinfo, refBase):
    indelBase = ["D", "I"]
    snpBase = ["A", "T", "C", "G"]
    basicBase = ["A", "T", "C", "G", "D", "I"]
    countDic = {"A": 0, "T": 0, "C": 0, "G": 0, "D": 0, "I": 0}
    check_base = list(set(basicBase).difference(set([refBase])))
    sitemutinfoList = sitemutinfo.split(":")
    countDic['A'] += int(sitemutinfoList[0])
    countDic['T'] += int(sitemutinfoList[1])
    countDic['C'] += int(sitemutinfoList[2])
    countDic['G'] += int(sitemutinfoList[3])
    countDic['D'] += int(sitemutinfoList[4])
    countDic['I'] += int(sitemutinfoList[5])
    gtList = [i for i in check_base if countDic[i] != 0]
    indelintersection = list(set(gtList).intersection(set(indelBase)))
    snpintersection = list(set(gtList).intersection(set(snpBase)))
    gtListAllele = [i for i in check_base if countDic[i] > 1]
    gtListAlleleEqul1 = [i for i in check_base if countDic[i] == 1]
    allelenum = len(gtListAllele)
    allelereadnum = sum(countDic[base] for base in gtListAllele)
    altGT = ",".join(list(i + "[" + str(countDic[i]) + "]" for i in list(set(snpintersection + indelintersection))))
    mutypeindel = "None"
    mutypesnp = "None"
    allelecheck = "None"
    readsupportcheck = "PASS_reads"
    if len(indelintersection) >= 1:
        mutypeindel = "INDEL"
    if len(snpintersection) >= 1:
        mutypesnp = "SNP"
    if len(gtListAlleleEqul1)/(len(gtListAlleleEqul1) + allelereadnum) >= 0.2:
        if len(gtListAlleleEqul1) + allelereadnum >= 2:
            allelecheck = "MultiAlleles"
    else:
        if len(gtListAlleleEqul1) >= 2 or allelenum >= 2:
            allelecheck = "MultiAlleles"
    if allelenum == 0:
        readsupportcheck = "NonePASS_reads"
    return mutypeindel, mutypesnp, allelecheck, readsupportcheck, altGT

# test assign_mutype_snpindex
sitemutinfo = "10:10:0:1:0:0"
refBase = "A"
print(assign_mutype_snpindex(sitemutinfo, refBase))

('None', 'SNP', 'None', 'PASS_reads', 'T[10],G[1]')


In [69]:
def assign_mutype_snpindex(sitemutinfo, refBase):
    indelBase = {"D", "I"}
    snpBase = {"A", "T", "C", "G"}
    basicBase = {"A", "T", "C", "G", "D", "I"}
    countDic = {base: 0 for base in basicBase}
    check_base = basicBase - {refBase}
    sitemutinfoList = list(map(int, sitemutinfo.split(":")))
    for base, count in zip(["A", "T", "C", "G", "D", "I"], sitemutinfoList):
        countDic[base] += count
    gtList = [base for base in check_base if countDic[base] != 0]
    indelintersection = indelBase & set(gtList)
    snpintersection = snpBase & set(gtList)
    gtListAllele = [base for base in check_base if countDic[base] > 1]
    gtListAlleleEqul1 = [base for base in check_base if countDic[base] == 1]
    allelenum = len(gtListAllele)
    allelereadnum = sum(countDic[base] for base in gtListAllele)
    altGT = ",".join(f"{base}[{countDic[base]}]" for base in snpintersection | indelintersection)
    mutypeindel = "INDEL" if indelintersection else "None"
    mutypesnp = "SNP" if snpintersection else "None"
    allelecheck = "MultiAlleles" if (len(gtListAlleleEqul1) / (len(gtListAlleleEqul1) + allelereadnum) >= 0.2 and len(gtListAlleleEqul1) + allelereadnum >= 2) or (len(gtListAlleleEqul1) >= 2 or allelenum >= 2) else "None"
    readsupportcheck = "PASS_reads" if allelenum != 0 else "NonePASS_reads"
    return mutypeindel, mutypesnp, allelecheck, readsupportcheck, altGT

# test assign_mutype_snpindex
sitemutinfo = "1:2:4:5:8:9"
refBase = "A"
print(assign_mutype_snpindex(sitemutinfo, refBase))


('INDEL', 'SNP', 'MultiAlleles', 'PASS_reads', 'T[2],C[4],G[5],I[9],D[8]')


In [23]:
import  scipy.stats as stats
def split_baseQua_lowqualitynum(refBase, oneCellBaseQua, lowBaseCutoff):
    # Example: 32:32||||20|
    basicBase = ["A", "T", "C", "G", "D", "I"]
    baseQuainfoList = oneCellBaseQua.split("|")
    lowrefBaseNum = len([int(i) for i in baseQuainfoList[basicBase.index(refBase)].split(":") if i and int(i) <= lowBaseCutoff])
    altBase = [base for base in basicBase if base != refBase and baseQuainfoList[basicBase.index(base)]]
    lowaltBaseNum = sum(len([int(j) for j in baseQuainfoList[basicBase.index(base)].split(":") if j and int(j) <= lowBaseCutoff]) for base in altBase)
    return lowrefBaseNum, lowaltBaseNum

# test split_baseQua_lowqualitynum
refBase = "A"
oneCellBaseQua = "32:32:32:32:30:30|21:12||||"
lowBaseCutoff = 20
lowwtBasenum, lowmutBasenum = split_baseQua_lowqualitynum(refBase, oneCellBaseQua, lowBaseCutoff)
altReadsnum = 1
totalReadsnum = 7
p_fisher_lowBase = stats.fisher_exact([[int(lowmutBasenum), int(lowwtBasenum)], [int(altReadsnum) - int(lowmutBasenum), int(totalReadsnum) - int(altReadsnum) - int(lowwtBasenum)]]).pvalue
print(p_fisher_lowBase)
print([int(lowmutBasenum), int(lowwtBasenum)], [int(altReadsnum) - int(lowmutBasenum), int(totalReadsnum) - int(altReadsnum) - int(lowwtBasenum)])

0.14285714285714285
[1, 0] [0, 6]


In [2]:
import collections
# count base number exclude other base
def count_base_fromDic(baseDic, excludeBaseList):
    """
    Counts the number of bases in a dictionary, excluding specified bases.
    Args:
        baseDic (dict): Dictionary containing base counts.
        excludeBaseList (list): List of bases to exclude from the count.
    Returns:
        int: The count of bases excluding the specified bases.
    """
    check_bases = list(set(baseDic.keys()).difference(set(excludeBaseList)))
    ALT_NUM = sum(baseDic[base] for base in check_bases)
    return ALT_NUM

# return cell mutations info
def get_mut_bycell(CELL_COUNTS_DIC):
    """
    Returns the mutation information for each cell based on the provided dictionary of cell counts.

    Args:
        CELL_COUNTS_DIC (dict): Dictionary containing cell counts for each base type.

    Returns:
        tuple: A tuple containing:
            - cellOut (list): List of unique cell identifiers.
            - countOut (list): List of mutation counts for each cell in the format "A|T|C|G|D|I".
    """
    totalCells = set(CELL_COUNTS_DIC["A"] + CELL_COUNTS_DIC["T"] + CELL_COUNTS_DIC["C"] + CELL_COUNTS_DIC["G"] + CELL_COUNTS_DIC["D"] + CELL_COUNTS_DIC["I"])
    cellOut = []
    countOut = []
    ca = collections.Counter(CELL_COUNTS_DIC["A"])
    ct = collections.Counter(CELL_COUNTS_DIC["T"])
    cc = collections.Counter(CELL_COUNTS_DIC["C"])
    cg = collections.Counter(CELL_COUNTS_DIC["G"])
    cd = collections.Counter(CELL_COUNTS_DIC["D"])
    ci = collections.Counter(CELL_COUNTS_DIC["I"])
    for cell in totalCells:
        num_a = ca.get(cell, 0)
        num_t = ct.get(cell, 0)
        num_c = cc.get(cell, 0)
        num_g = cg.get(cell, 0)
        num_d = cd.get(cell, 0)
        num_i = ci.get(cell, 0)
        count_info = "|".join([str(num_a), str(num_t), str(num_c), str(num_g), str(num_d), str(num_i)])
        cellOut.append(cell)
        countOut.append(count_info)
    return cellOut, countOut

def get_mut_total(CELL_COUNTS_DIC, refBase):
    """
    Calculate the total number of mutations and cell counts for each base type.
    Args:
        CELL_COUNTS_DIC (dict): Dictionary containing cell counts for each base type.
        refBase (str): Reference base to exclude from the count.
    Returns:
        tuple: A tuple containing:
            - countCellNumSTR (str): String representation of cell counts for each base type.
            - countAltNumSTR (str): String representation of alternative base counts for each base type.
            - countCellNum (int): Total number of cells.
            - countAltNum (int): Total number of alternative bases.
            - countAltCellNum (int): Total number of cells with alternative bases.
    """
    basic_base = ["A", "T", "C", "G", "D", "I"]
    CELL_COUNTS_DIC_NEW_0 = {x: len(set(CELL_COUNTS_DIC[x])) for x in basic_base}
    CELL_COUNTS_DIC_NEW_1 = {x: len(CELL_COUNTS_DIC[x]) for x in basic_base}
    print(CELL_COUNTS_DIC_NEW_0)
    print(CELL_COUNTS_DIC_NEW_1)
    countCellNumSTR = ':'.join([str(CELL_COUNTS_DIC_NEW_0[x]) for x in basic_base])
    countAltNumSTR = ':'.join([str(CELL_COUNTS_DIC_NEW_1[x]) for x in basic_base])
    countCellNum = sum(CELL_COUNTS_DIC_NEW_0.values())
    countAltNum = count_base_fromDic(CELL_COUNTS_DIC_NEW_1, [refBase])
    countAltCellNum = count_base_fromDic(CELL_COUNTS_DIC_NEW_0, [refBase])
    return countCellNumSTR, countAltNumSTR, countCellNum, countAltNum, countAltCellNum

CELL_COUNTS_DIC = {"A": ["C1", "C1", "C2"], "T": ["C3", "C3", "C4"], "C": [], "G": [], "N": [], "D": [], "I": []}
REF_BASE = "A"
COUNTINFOCELL, COUNTINFOALT, COUNTOTALCELL, COUNTOTALALT, COUNTOTALCELLALT = get_mut_total(CELL_COUNTS_DIC, REF_BASE)
CELLUNIQ, CELLMUTCOUNTINFO = get_mut_bycell(CELL_COUNTS_DIC)
print(COUNTINFOCELL, COUNTINFOALT, COUNTOTALCELL, COUNTOTALALT, COUNTOTALCELLALT)
print(CELLUNIQ, CELLMUTCOUNTINFO)

{'A': 2, 'T': 2, 'C': 0, 'G': 0, 'D': 0, 'I': 0}
{'A': 3, 'T': 3, 'C': 0, 'G': 0, 'D': 0, 'I': 0}
2:2:0:0:0:0 3:3:0:0:0:0 4 3 2
['C4', 'C2', 'C3', 'C1'] ['0|1|0|0|0|0', '1|0|0|0|0|0', '0|2|0|0|0|0', '2|0|0|0|0|0']


In [ ]:

lowwtBasenum, lowmutBasenum = split_baseQua_lowqualitynum(refBase, integrate_baseQua, lowBaseCutoff)
# fisher test for lowQualityBases
p_fisher_lowBase = stats.fisher_exact([[int(lowmutBasenum), int(lowwtBasenum)], [int(altReadsnum) - int(lowmutBasenum), int(totalReadsnum) - int(altReadsnum) - int(lowwtBasenum)]]).pvalue
